# ROCF VLM · 복합도형 그리기 과정으로 아밀로이드 베타 예측하기 (합성 데이터 예제)

이 노트북은 사전 생성된 **블라인드 합성 ROCF 데이터셋**(`data.zip`, `N=400`)의 파일 구조를 살펴보고, 정적 기준선 실행, 5 Hz 그리기 과정 시각화, 그리고 학습 데이터 교차검증을 직접 실행해 보는 입문용 안내서입니다.

- **ROCF(Rey-Osterrieth Complex Figure)** 는 18개 도형 요소로 이루어진 복합도형을 보고 따라 그리는 **Copy** 과제와, 잠시 뒤 기억에만 의존해 다시 그리는 **Delayed Recall** 과제로 구성됩니다.
- **학습(`train`, 280명)** 에서는 완성된 정적 흑백 그림(`images/*.png`)과 5 Hz 펜 궤적(`strokes/*.csv`)뿐 아니라, `train_privileged.csv`의 뇌영상·임상 지표(`hippocampal_md`, `precuneus_md`, `hippocampal_vol`, `amyloid_centiloid` 등)와 36점 채점표를 모두 활용할 수 있습니다.
- **평가(`test`, 120명)** 에서는 **ROCF 그림(`images/*.png`)과 5 Hz 펜 궤적(`strokes/*.csv`)만** 주어지며 정답 라벨은 비공개입니다.

먼저 터미널에서 제공된 압축 파일을 풀어 두거나(또는 아래 첫 코드 셀이 `data.zip`을 자동 해제합니다):
```bash
python -m zipfile -e data.zip .
```


In [ ]:
from pathlib import Path
import json
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import rocf_viz
from starter_pipeline import extract_static_image_features, run_pipeline as run_static_starter

DATA_DIR = Path("data")
if not (DATA_DIR / "participants.csv").is_file() and Path("data.zip").is_file():
    with zipfile.ZipFile("data.zip", "r") as zf:
        zf.extractall(".")

part = pd.read_csv(DATA_DIR / "participants.csv", dtype={"subject_id": str})
priv = pd.read_csv(DATA_DIR / "train_privileged.csv", dtype={"subject_id": str})

print("participants.csv 열:", list(part.columns), "| 행 수:", len(part))
print("split 분포:\n", part["split"].value_counts().to_dict())
print("\ntrain_privileged.csv 열:", list(priv.columns))
print("test 참가자가 train_privileged에 포함되었는가?",
      len(set(part.loc[part["split"] == "test", "subject_id"]) & set(priv["subject_id"])) > 0)
priv.head(4)


## 1. 학습 데이터 탐색: 인지 상태·도형 점수·뇌 지표 비교

`train_privileged.csv`에는 인지정상(`CU`)과 경도인지장애(`MCI`) 참가자의 진단 정보, 36점 기준 도형 채점(`copy_score_36`, `recall_score_36`), 그리고 아밀로이드 침착 정도(`amyloid_centiloid`)와 해마·뇌피질 미세구조 지표(`hippocampal_md`, `hippocampal_vol`, `precuneus_md` 등)가 담겨 있습니다. 각 집단에서 분포가 어떻게 다른지 확인해 봅시다.


In [ ]:
summary_tbl = priv.groupby("diagnosis").agg(
    n=("subject_id", "count"),
    ab_pos_rate=("amyloid_pos", "mean"),
    copy_mean=("copy_score_36", "mean"),
    recall_mean=("recall_score_36", "mean"),
    hipp_md_mean=("hippocampal_md", "mean"),
    hipp_vol_mean=("hippocampal_vol", "mean"),
).round(3)
print(summary_tbl)

print("\nCU 내부 아밀로이드 상태별 평균 비교:")
print(priv[priv["diagnosis"] == "CU"].groupby("amyloid_pos")[
    ["copy_score_36", "recall_score_36", "amyloid_centiloid", "hippocampal_md", "precuneus_md", "hippocampal_vol"]
].mean().round(4))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for dx, col in [("CU", "#2a6fdb"), ("MCI", "#d95f02")]:
    sub = priv[priv["diagnosis"] == dx]
    axes[0].hist(sub["copy_score_36"], bins=16, alpha=0.65, label=dx, color=col)
    axes[1].hist(sub["recall_score_36"], bins=16, alpha=0.65, label=dx, color=col)
axes[0].set_title("ROCF Copy Score (/36)")
axes[0].set_xlabel("Copy Score (/36)"); axes[0].legend()
axes[1].set_title("ROCF Delayed Recall Score (/36)")
axes[1].set_xlabel("Recall Score (/36)"); axes[1].legend()

cu = priv[priv["diagnosis"] == "CU"]
for ab, lbl, col in [(0, "CU Ab-", "#2b8cbe"), (1, "CU Ab+", "#e34a33")]:
    s = cu[cu["amyloid_pos"] == ab]
    axes[2].scatter(s["hippocampal_vol"], s["hippocampal_md"], alpha=0.7, s=26, label=lbl, color=col)
axes[2].set_title("Brain Markers in CU (Train Only)")
axes[2].set_xlabel("Hippocampal Volume (mL)")
axes[2].set_ylabel("Hippocampal MD (1e-3 mm2/s)")
axes[2].legend()
plt.tight_layout()
plt.show()


**생각해 보기:** `test` 단계에서는 뇌영상·임상 지표를 사용할 수 없지만, `train` 단계에서는 `hippocampal_md`, `precuneus_md`, `amyloid_centiloid` 같은 학습 전용 정보를 모두 볼 수 있습니다. 이러한 특권 정보 활용 학습(Learning Using Privileged Information) 전략을 모델 훈련에 녹여 내면 `test` 추론 성능을 높이는 데 어떤 도움을 줄 수 있을까요?

## 2. 정적 그림 기준선 실행 및 제출 규격 확인

`starter_pipeline.py`는 완성된 흑백 이미지(`images/*.png`)만 입력으로 사용해 아밀로이드 양성을 예측하는 간단한 기준선입니다. 실행 결과로 `predictions.csv`(`subject_id`, `y_prob` 두 열)와 최소 규격의 `summary.json`(`claim_signal`, `method_summary`)이 생성되는지 확인해 봅시다.


In [ ]:
out_starter = Path("outputs_notebook_starter")
summary_starter = run_static_starter(DATA_DIR, out_starter, requested_device="cpu", seed=42)
print("summary.json:")
print(json.dumps(summary_starter, ensure_ascii=False, indent=2))

pred_starter = pd.read_csv(out_starter / "predictions.csv", dtype={"subject_id": str})
test_ids = set(part.loc[part["split"] == "test", "subject_id"])
assert list(pred_starter.columns) == ["subject_id", "y_prob"]
assert set(pred_starter["subject_id"]) == test_ids
assert pred_starter["y_prob"].notna().all()
assert pred_starter["y_prob"].between(0.0, 1.0).all()
print(f"\npredictions.csv 확인 완료: {pred_starter.shape[0]}행 x {pred_starter.shape[1]}열, "
      f"y_prob 범위=[{pred_starter['y_prob'].min():.4f}, {pred_starter['y_prob'].max():.4f}]")
pred_starter.head(5)


## 3. 정적 그림과 5 Hz 그리기 과정 패널 비교하기

`strokes/*.csv`에는 0.2초 간격(5 Hz)으로 기록된 `t_sec`, `x_mm`, `y_mm`, `pen_down` 좌표가 들어 있습니다. `rocf_viz.render_process_panel`은 이 궤적을 **시간 순서 색상**(파란색=초반 -> 빨간색=후반), **펜을 떼고 멈춘 지점**(보라색 고리), **진행 단계별 누적 스냅샷**(25%, 50%, 75%, 100%)으로 한눈에 보여 줍니다. `CU` 집단에서 도형 점수가 높은 `Aβ-` 참가자와 `Aβ+` 참가자 1명씩을 골라 정적 흑백 그림과 비교해 봅시다.


In [ ]:
cu_df = priv[priv["diagnosis"] == "CU"].copy()
cu_df["total_36"] = cu_df["copy_score_36"] + cu_df["recall_score_36"]
cu_high = cu_df[(cu_df["copy_score_36"] >= 35.0) & (cu_df["recall_score_36"] >= 34.0)]
neg_pool = cu_high[cu_high["amyloid_pos"] == 0]
if neg_pool.empty:
    neg_pool = cu_df[cu_df["amyloid_pos"] == 0].sort_values("total_36", ascending=False)
pos_pool = cu_high[cu_high["amyloid_pos"] == 1]
if pos_pool.empty:
    pos_pool = cu_df[cu_df["amyloid_pos"] == 1].sort_values("total_36", ascending=False)
sid_neg = str(neg_pool["subject_id"].iloc[0])
sid_pos = str(pos_pool["subject_id"].iloc[0])
print("비교 대상:", sid_neg, "(CU Ab-) vs", sid_pos, "(CU Ab+)")

sample_copy = pd.read_csv(DATA_DIR / "strokes" / f"{sid_pos}_copy.csv")
print("5 Hz 궤적 CSV 앞부분 (t_sec, x_mm, y_mm, pen_down):")
print(sample_copy.head(5))

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for r_idx, (sid, tag) in enumerate([(sid_neg, "CU Ab-"), (sid_pos, "CU Ab+")]):
    static_rec = Image.open(DATA_DIR / "images" / f"{sid}_recall.png")
    c_df = pd.read_csv(DATA_DIR / "strokes" / f"{sid}_copy.csv")
    r_df = pd.read_csv(DATA_DIR / "strokes" / f"{sid}_recall.csv")
    panel = rocf_viz.render_process_panel(c_df, r_df)

    axes[r_idx, 0].imshow(static_rec, cmap="gray")
    axes[r_idx, 0].set_title(f"{sid} ({tag}) - Static Recall PNG")
    axes[r_idx, 0].axis("off")

    axes[r_idx, 1].imshow(panel)
    axes[r_idx, 1].set_title(f"{sid} ({tag}) - 5 Hz Color-Time & Progress Panel")
    axes[r_idx, 1].axis("off")

plt.tight_layout()
plt.show()


### (선택) OpenRouter VLM 탐색 훅 안내

개발 단계에서는 위 `render_process_panel` 이미지를 소수 표본에 대해 OpenRouter VLM API로 보내 시각 패턴과 오류 사례를 탐색할 수 있습니다. 아래 셀은 기본값(`USE_VLM = False`)으로 오프라인 상태를 유지하며, **최종 제출용 `run_pipeline.py` 본체는 API 키 없이 오프라인으로 단독 실행**되어야 합니다.

- 실행 전 터미널에서 `export OPENROUTER_API_KEY="sk-or-..."`처럼 환경변수로만 키를 설정하세요.
- **보안 경고:** 실제 API 키 문자열을 노트북 코드 셀에 직접 적거나, 출력 셀에 키·응답 흔적이 남은 채로 Git에 커밋·공유하지 마세요.


In [ ]:
# 절대 OPENROUTER_API_KEY 문자열을 코드에 직접 적지 말고, 터미널에서 export OPENROUTER_API_KEY="sk-or-..."로 설정하세요.
USE_VLM = False  # 시각 탐색이 필요할 때만 True로 바꾸고 환경변수를 설정하세요
api_key = os.environ.get("OPENROUTER_API_KEY", "").strip()
if USE_VLM and not api_key:
    print("경고: USE_VLM=True이지만 OPENROUTER_API_KEY 환경변수가 비어 있습니다. 코드에 키를 직접 적지 말고 터미널에서 export OPENROUTER_API_KEY=\"sk-or-...\" 후 다시 실행하세요.")
elif USE_VLM and api_key:
    import base64, io, urllib.request
    c_df = pd.read_csv(DATA_DIR / "strokes" / f"{sid_pos}_copy.csv")
    r_df = pd.read_csv(DATA_DIR / "strokes" / f"{sid_pos}_recall.csv")
    buf = io.BytesIO()
    rocf_viz.render_process_panel(c_df, r_df).save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    payload = {
        "model": "google/gemini-2.5-flash",
        "temperature": 0.0,
        "max_tokens": 200,
        "messages": [{"role": "user", "content": [
            {"type": "text", "text": "Inspect Copy vs Delayed Recall stroke order (blue=early, red=late) and purple pause rings. Summarize structural-junction pauses and skeleton-first order in 3 bullet points."},
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
        ]}],
    }
    req = urllib.request.Request(
        "https://openrouter.ai/api/v1/chat/completions",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        print(json.loads(resp.read().decode("utf-8"))["choices"][0]["message"]["content"])
else:
    print("오프라인 기본 모드(USE_VLM=False): VLM API 호출을 건너뜁니다.")


## 4. 학습 데이터 교차검증으로 기준선 성능 점검하기 (진단군 내 층화 평가)

블라인드 `test`(120명) 정답은 비공개이므로, 모델 개발과 성능 검증은 `train`(280명) 내부 교차검증과 자체 대조 검증으로 수행합니다. 정적 흑백 그림만 사용하는 기준선을 5-폴드 층화 교차검증으로 평가해, 전체 통합 AUROC와 진단군별(`CU`, `MCI`, `within_dx_auroc`) 성능이 어떻게 다른지 비교해 봅시다.


In [ ]:
train_ids = priv["subject_id"].astype(str).tolist()
y_tr = priv["amyloid_pos"].to_numpy(dtype=int)
dx_tr = priv["diagnosis"].astype(str).to_numpy()
strata = priv["diagnosis"].astype(str) + "_" + priv["amyloid_pos"].astype(str)

images_dir = DATA_DIR / "images"
x_tr = np.vstack([
    extract_static_image_features(images_dir / f"{sid}_copy.png", images_dir / f"{sid}_recall.png")
    for sid in train_ids
])

oof_prob = np.zeros(len(train_ids), dtype=float)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for tr_idx, va_idx in cv.split(x_tr, strata):
    clf = Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("scl", StandardScaler()),
        ("lr", LogisticRegression(C=0.5, max_iter=500, random_state=42)),
    ])
    clf.fit(x_tr[tr_idx], y_tr[tr_idx])
    oof_prob[va_idx] = clf.predict_proba(x_tr[va_idx])[:, 1]

cv_metrics = rocf_viz.compute_within_dx_metrics(y_tr, oof_prob, dx_tr)
print("정적 기준선 5-폴드 교차검증 성능(train N=280):", cv_metrics)


## 정리해 보세요

1. 전체 표본의 AUROC만 볼 때와 인지정상(`CU`)·경도인지장애(`MCI`) 각 진단군 내부 성능을 함께 볼 때 어떤 차이가 나타나며, 왜 두 지표를 같이 확인해야 할까요?
2. `train_privileged.csv`의 뇌영상·임상 지표(`hippocampal_md`, `precuneus_md`, `hippocampal_vol`, `amyloid_centiloid`)를 학습 때만 활용하고 `test` 추론에는 넣지 않으려면 어떤 모델링 전략이 효과적일까요?
3. 정적 흑백 그림(`images/*.png`)과 5 Hz 그리기 과정 패널(`render_process_panel`)에서 관찰되는 시각적 차이는 무엇이며, 모델이 우연한 지름길이나 배경 차이에 의존하지 않는지 어떻게 스스로 점검할 수 있을까요?

**제출 규격 자동 점검:**
```bash
python smoke_test.py --pipeline starter_pipeline.py --device cpu
```
